# 🧠 EX61: รูปแบบชุดข้อมูล YOLO (Dataset Format & Coordinate Conversion)

แต่ละภาพมีไฟล์ `.txt` label ที่ใช้พิกัด **normalized**

## คณิตศาสตร์
กำหนด pixel bbox $[x_{min}, y_{min}, x_{max}, y_{max}]$ บนภาพขนาด $W\times H$:
$$x_c = \frac{x_{min}+x_{max}}{2W},\quad y_c = \frac{y_{min}+y_{max}}{2H},\quad w = \frac{x_{max}-x_{min}}{W},\quad h = \frac{y_{max}-y_{min}}{H}$$

บรรทัด label: `<class_id> <x_c> <y_c> <w> <h>` (ค่าทั้งหมดอยู่ใน [0,1])

## data.yaml
```yaml
path: /abs/path/dataset
train: train/images
val:   val/images
nc: 3
names: {0: cat, 1: dog, 2: bird}
```

## 🔗 ลิงก์
- [[EX62_Inference_Visualization_TH]] | [[YOLO_Learning_Plan]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from solution import convert_to_yolo_format
%matplotlib inline

W, H = 640, 480
test_cases = [
    ([120, 160, 360, 400], 0, "cat"),
    ([400, 50, 600, 250], 1, "dog"),
    ([10, 10, 100, 80],   2, "bird"),
]

print("\n--- เริ่มการตรวจสอบ ---")
label_lines = []
for bbox, class_id, name in test_cases:
    yolo_str = convert_to_yolo_format(bbox, W, H, class_id)
    p = yolo_str.split()
    cid, xc, yc, bw, bh = int(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4])
    assert 0<=xc<=1 and 0<=yc<=1 and 0<=bw<=1 and 0<=bh<=1, f"พิกัดเกินขอบเขต {name}!"
    print(f"  {name} | pixel={bbox}")
    print(f"    YOLO: {yolo_str}")
    print(f"    ตรวจสอบ: xc={xc:.4f} yc={yc:.4f} w={bw:.4f} h={bh:.4f} ✅")
    label_lines.append(yolo_str)

label_path = Path("audit_label.txt")
label_path.write_text("\n".join(label_lines) + "\n")
print(f"\n✅ เขียน {label_path}: {label_path.stat().st_size} bytes")
print(f"เนื้อหา:\n{label_path.read_text()}")

print("แปลงกลับ (round-trip):")
for line in label_path.read_text().strip().split("\n"):
    p = line.split()
    cid2, xc2, yc2, bw2, bh2 = int(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4])
    x1=(xc2-bw2/2)*W; y1=(yc2-bh2/2)*H; x2=(xc2+bw2/2)*W; y2=(yc2+bh2/2)*H
    print(f"  class={cid2} → pixel xyxy=[{x1:.1f},{y1:.1f},{x2:.1f},{y2:.1f}]")
print("--- สิ้นสุดการตรวจสอบ ---")

fig, ax = plt.subplots(figsize=(8,6))
ax.imshow(np.ones((H,W,3), dtype=np.uint8)*230)
colors = ["#e74c3c","#3498db","#2ecc71"]
for (bbox,cid,name),color in zip(test_cases, colors):
    x1,y1,x2,y2 = bbox
    ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1, linewidth=2, edgecolor=color, facecolor=color, alpha=0.25))
    ax.text(x1,y1-5, name, color=color, fontsize=10, fontweight="bold")
ax.set_xlim(0,W); ax.set_ylim(H,0)
ax.set_xlabel("X (px)"); ax.set_ylabel("Y (px)")
ax.set_title("การแปลงพิกัด YOLO Dataset Format")
plt.tight_layout(); plt.show()

label_path.unlink(missing_ok=True)
